# PEPITA on ResNet-20 / CIFAR-10: results from the stored weights

This notebook scores the stored weight files on the noisy CIFAR-10 test set and writes the results to an Excel workbook.
It trains nothing and reads no other notebook, script or results file: everything it needs is in the cells below.

| Row | Weight file, for input-noise variance *v* |
|---|---|
| Init (no retrain) | `weights_baselines/full_retrain_var_0.0.pth`, the clean model every run starts from |
| PEPITA | `weights_pepita_resnet20_cifar10/pepita_var_<v>.pth`, a ResNet-20 state dict (PEPITA adapts a plain ResNet-20) |

- **Test set:** the official CIFAR-10 test set, 10,000 images, no re-split. Gaussian noise of variance *v* is added
  after normalisation, drawn on the CPU with seed 42 at every level.
- **Model:** ResNet-20 in eval mode, so BatchNorm uses the running statistics stored in each file; non-finite logits, if
  any, are set to 0.
- **Metrics:** accuracy, macro precision, recall and F1 over the 10 classes, and mAP (the mean one-vs-rest average
  precision on the softmax scores), rounded to 6 decimals. Accuracies can differ by a few test images between machines
  (GPU and CPU arithmetic).
- **Output:** `resnet20_cifar10/results_pepita_resnet20_cifar10/results_pepita_resnet20_cifar10_reproduced.xlsx`, one
  sheet `long` with a row per noise level and method; a re-run overwrites it. The name carries no `YYYYmmdd_HHMMSS`
  stamp, so the plotting notebooks, which read the workbook with the newest stamp in each results folder, keep using the
  reported one.

**How to run:** run all cells. The notebook looks for the project folder (the one holding `resnet20_cifar10/` and
`data/`) from its own folder upwards and reads CIFAR-10 from `data/cifar-10-batches-py`, downloading it there if it is
missing. It needs torch, torchvision, numpy, pandas and openpyxl.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

## Settings

In [2]:
# ---- settings: edit here if needed ----
DATASET = "resnet20_cifar10"
METHOD_KEY = "pepita"                  # the weights folder is weights_<METHOD_KEY>_resnet20_cifar10/
METHOD_LABEL = "PEPITA"                # the method's row label in the workbook

DEVICE = "cpu"                         # "cuda" works too, on a GPU this torch build supports
BATCH_SIZE = 1024                      # evaluation batch; lower it if memory is short
SEED = 42
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]   # input-noise variances
TEST_NOISE_SEED = SEED                 # the noise stream of the PEPITA sweep
NUM_CLASSES = 10


def find_project_root(start=Path.cwd()):
    """The BOLSTER folder: the first folder, from the notebook's folder upwards, that holds DATASET/ and data/."""
    for c in (start, *start.parents):
        if (c / DATASET).is_dir() and (c / "data").is_dir():
            return c
    raise FileNotFoundError(f"no folder with {DATASET}/ and data/ above {start}")


PROJECT = find_project_root()
DATA_ROOT = PROJECT / "data"
BASELINE_WEIGHTS = PROJECT / DATASET / "weights_baselines"
METHOD_WEIGHTS = PROJECT / DATASET / f"weights_{METHOD_KEY}_{DATASET}"
RESULTS_DIR = PROJECT / DATASET / f"results_{METHOD_KEY}_{DATASET}"
# no YYYYmmdd_HHMMSS stamp in the name: the plotting notebooks keep reading the reported workbook
XLSX_PATH = RESULTS_DIR / f"results_{METHOD_KEY}_{DATASET}_reproduced.xlsx"

print(f"project: {PROJECT}\ndevice: {DEVICE}")

project: /home/rahman2/GIT/BOLSTER
device: cpu


## The network

In [3]:
class BasicBlock(nn.Module):
    """Two 3x3 conv-BN layers with an identity or 1x1-projection shortcut (ResNet v1)."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.shortcut = (nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
                         if (stride != 1 or in_ch != out_ch) else nn.Identity())

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(y + self.shortcut(x))


class ResNet20(nn.Module):
    """ResNet-20 for CIFAR (He et al. 2015): a 3x3 stem, three stages of 3 blocks (16, 32, 64 channels; feature maps
    32x32 -> 16x16 -> 8x8), global average pooling and a linear head. The module names match the stored state dicts."""
    def __init__(self, num_classes=NUM_CLASSES, n=3):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.stage1 = self._make_stage(16, 16, n, stride=1)
        self.stage2 = self._make_stage(16, 32, n, stride=2)
        self.stage3 = self._make_stage(32, 64, n, stride=2)
        self.fc     = nn.Linear(64, num_classes)

    @staticmethod
    def _make_stage(in_ch, out_ch, blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        layers += [BasicBlock(out_ch, out_ch) for _ in range(blocks - 1)]
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.stage3(self.stage2(self.stage1(x)))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.fc(x)

## Metrics

In [4]:
def macro_prf(y_true, y_pred, num_classes=NUM_CLASSES):
    """Macro precision, recall and F1 over the classes; a class with an empty denominator contributes 0."""
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    den  = prec + rec
    f1   = np.divide(2 * prec * rec, den, out=np.zeros_like(tp), where=den > 0)
    return float(prec.mean()), float(rec.mean()), float(f1.mean())


def average_precision(is_pos, scores):
    """One-vs-rest average precision of one class (the scikit-learn definition: the sum over the distinct score
    thresholds of (R_n - R_{n-1}) P_n)."""
    n_pos = int(is_pos.sum())
    if n_pos == 0:
        return 0.0
    order = np.argsort(-scores, kind="mergesort")
    y, s = is_pos[order], scores[order]
    last = np.r_[np.where(np.diff(s))[0], y.size - 1]        # the last index of each distinct score
    tps = np.cumsum(y)[last]
    fps = (1 + last) - tps
    precision = np.r_[1.0, tps / (tps + fps)]
    recall    = np.r_[0.0, tps / n_pos]
    return float(np.sum(np.diff(recall) * precision[1:]))


def finite_logits(z):
    """Non-finite logits (a diverged model) set to 0, so that such a model scores like a uniform predictor."""
    return z if torch.isfinite(z).all() else torch.nan_to_num(z, nan=0.0, posinf=0.0, neginf=0.0)


@torch.no_grad()
def eval_metrics(state_dict, x, y, model_cls=ResNet20):
    """accuracy, macro precision / recall / F1 and mAP of the model in state_dict on (x, y). The argmax gives the
    predictions, the softmax the scores for mAP."""
    model = model_cls().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    probs, preds = [], []
    for i in range(0, y.numel(), BATCH_SIZE):
        xb = x[i:i + BATCH_SIZE].to(DEVICE)
        logits = finite_logits(model(xb))
        probs.append(F.softmax(logits, dim=1).cpu())
        preds.append(logits.argmax(1).cpu())
    probs, y_pred, y_true = torch.cat(probs).numpy(), torch.cat(preds).numpy(), y.numpy()
    precision, recall, f1 = macro_prf(y_true, y_pred)
    mAP = float(np.mean([average_precision((y_true == c).astype(np.int64), probs[:, c]) for c in range(NUM_CLASSES)]))
    return {"accuracy": float((y_pred == y_true).mean()), "precision": precision, "recall": recall, "f1": f1,
            "map": mAP}

## The test set

In [5]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])


def stack(ds):
    xs, ys = zip(*((x, y) for x, y in ds))
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)


def add_noise(x, var, seed):
    """x plus Gaussian noise of variance var, drawn on the CPU from a generator seeded with seed."""
    if var <= 0:
        return x
    g = torch.Generator().manual_seed(seed)
    return x + torch.randn(x.shape, generator=g) * float(np.sqrt(var))


x_test, y_test = stack(datasets.CIFAR10(str(DATA_ROOT), train=False, download=True, transform=tfm))
print(f"test set: the official CIFAR-10 test set, {y_test.numel():,} images")

test set: the official CIFAR-10 test set, 10,000 images


## Scores and workbook
Which weight file feeds which row; every row is scored on that level's noisy test set, then the workbook is written.

In [6]:
ROWS = [                               # (row label, weight file for noise variance v), in the workbook's order
    ("Init (no retrain)", lambda v: BASELINE_WEIGHTS / "full_retrain_var_0.0.pth"),
    (METHOD_LABEL, lambda v: METHOD_WEIGHTS / f"{METHOD_KEY}_var_{v}.pth"),
]
METRIC_COLS = ["accuracy", "precision", "recall", "f1", "map"]


def load_state(path):
    return torch.load(path, map_location="cpu", weights_only=True)


records = []
for v in NOISE_LEVELS:
    level = f"{v:.0%}"
    x_noisy = add_noise(x_test, v, seed=TEST_NOISE_SEED)
    for label, path_of in ROWS:
        path = path_of(v)
        r = eval_metrics(load_state(path), x_noisy, y_test)
        records.append({"noise_level": level, "method": label, **{m: round(r[m], 6) for m in METRIC_COLS}})
        print(f"{level:>5}  {label:<20} acc {r['accuracy']:.6f}  f1 {r['f1']:.6f}  map {r['map']:.6f}   <- {path.name}")

results = pd.DataFrame(records, columns=["noise_level", "method", *METRIC_COLS])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_excel(XLSX_PATH, sheet_name="long", index=False)
print(f"\nwrote {XLSX_PATH.relative_to(PROJECT)}")

  10%  Init (no retrain)    acc 0.367100  f1 0.381335  map 0.553013   <- full_retrain_var_0.0.pth


  10%  PEPITA               acc 0.631100  f1 0.630766  map 0.700250   <- pepita_var_0.1.pth


  20%  Init (no retrain)    acc 0.219900  f1 0.194432  map 0.347545   <- full_retrain_var_0.0.pth


  20%  PEPITA               acc 0.475300  f1 0.474098  map 0.506984   <- pepita_var_0.2.pth


  30%  Init (no retrain)    acc 0.184500  f1 0.135570  map 0.261980   <- full_retrain_var_0.0.pth


  30%  PEPITA               acc 0.397600  f1 0.394862  map 0.408128   <- pepita_var_0.3.pth


... (10 outputs left out)

  90%  Init (no retrain)    acc 0.157100  f1 0.096921  map 0.147719   <- full_retrain_var_0.0.pth


  90%  PEPITA               acc 0.275800  f1 0.270963  map 0.249200   <- pepita_var_0.9.pth


 100%  Init (no retrain)    acc 0.151000  f1 0.090931  map 0.142511   <- full_retrain_var_0.0.pth


 100%  PEPITA               acc 0.266400  f1 0.260680  map 0.240042   <- pepita_var_1.0.pth

wrote resnet20_cifar10/results_pepita_resnet20_cifar10/results_pepita_resnet20_cifar10_reproduced.xlsx
